# ReelScribe — Batch Transcription (Kaggle GPU)

**Шаги перед запуском:**
1. В левом меню Kaggle → **Secrets** → добавь `SUPABASE_URL` и `SUPABASE_ANON_KEY`
2. В правом меню → **Accelerator** → выбери **T4 GPU**
3. Нажми **Run All** (≈ 2–3 мин/рилс на T4)

In [ ]:
# ── Установка зависимостей ────────────────────────────────────────────────────
!pip install -q faster-whisper yt-dlp imageio-ffmpeg supabase deep-translator

In [ ]:
# ── Настройки ─────────────────────────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient

_s = UserSecretsClient()
SUPABASE_URL      = _s.get_secret("SUPABASE_URL")
SUPABASE_ANON_KEY = _s.get_secret("SUPABASE_ANON_KEY")

MODEL_NAME    = "large-v3"   # или "medium"
BATCH_LIMIT   = 200          # сколько джобов взять за один прогон
COOKIES_FILE  = None         # путь к cookies.txt если Instagram блокирует (/kaggle/input/...)
TRANSLATE     = True         # переводить не-русский текст на русский

In [ ]:
# ── Supabase ──────────────────────────────────────────────────────────────────
from supabase import create_client
db = create_client(SUPABASE_URL, SUPABASE_ANON_KEY)
print('Supabase OK')

In [ ]:
# ── Загрузка модели (GPU) ─────────────────────────────────────────────────────
import torch
from faster_whisper import WhisperModel

device       = 'cuda' if torch.cuda.is_available() else 'cpu'
compute_type = 'float16' if device == 'cuda' else 'int8'
print(f'Device: {device}, compute_type: {compute_type}')

model = WhisperModel(MODEL_NAME, device=device, compute_type=compute_type)
print(f'Модель {MODEL_NAME} загружена')

In [ ]:
# ── Функции ───────────────────────────────────────────────────────────────────
import os, random, tempfile, time, traceback
from pathlib import Path
import yt_dlp, imageio_ffmpeg

AUDIO_DIR = Path('/tmp/reelscribe_audio')
AUDIO_DIR.mkdir(parents=True, exist_ok=True)


def download_audio(url: str) -> Path:
    ffmpeg_dir = os.path.dirname(imageio_ffmpeg.get_ffmpeg_exe())
    opts = {
        'format': 'bestaudio/best',
        'outtmpl': str(AUDIO_DIR / '%(id)s.%(ext)s'),
        'quiet': True,
        'ffmpeg_location': ffmpeg_dir,
        'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'wav', 'preferredquality': '0'}],
        'postprocessor_args': {'ffmpeg': ['-ar', '16000', '-ac', '1']},
        'retries': 3,
    }
    if COOKIES_FILE:
        opts['cookiefile'] = COOKIES_FILE

    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(url, download=True)

    vid = info.get('id', '')
    wavs = list(AUDIO_DIR.glob(f'{vid}*.wav')) or sorted(AUDIO_DIR.glob('*.wav'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not wavs:
        raise FileNotFoundError(f'WAV не найден для {url}')
    return wavs[0], info


def transcribe(wav_path: Path) -> tuple[str, str, float]:
    segments, info = model.transcribe(str(wav_path), beam_size=5)
    text = ' '.join(s.text.strip() for s in segments).strip()
    return text, info.language, info.duration


def translate_to_ru(text: str) -> str:
    from deep_translator import GoogleTranslator
    chunks, size, buf = [], 4500, []
    for word in text.split():
        if sum(len(w) for w in buf) + len(word) > size:
            chunks.append(' '.join(buf)); buf = []
        buf.append(word)
    if buf: chunks.append(' '.join(buf))
    tr = GoogleTranslator(source='auto', target='ru')
    return ' '.join(tr.translate(c) for c in chunks)


def cleanup():
    for f in AUDIO_DIR.glob('*'):
        try: f.unlink()
        except: pass


print('Функции готовы')

In [ ]:
# ── Получаем очередь ──────────────────────────────────────────────────────────
from datetime import datetime, timezone

now = datetime.now(timezone.utc).replace(tzinfo=None).isoformat()
resp = (
    db.table('jobs')
    .select('*')
    .eq('state', 'queued')
    .lte('next_attempt_at', now)
    .order('next_attempt_at')
    .limit(BATCH_LIMIT)
    .execute()
)
jobs = resp.data
print(f'Джобов в очереди: {len(jobs)}')

In [ ]:
# ── Основной цикл ─────────────────────────────────────────────────────────────
from datetime import timedelta

MAX_ATTEMPTS = 3
done_count, fail_count = 0, 0

for job in jobs:
    job_id    = job['id']
    reel_id   = job['reel_id']
    session_id= job.get('session_id') or ''
    attempts  = job.get('attempts', 0) + 1

    # получаем URL рилса и флаги сессии
    reel_url = db.table('reels').select('url').eq('id', reel_id).execute().data[0]['url']
    do_translate = TRANSLATE
    if session_id:
        sess = db.table('import_sessions').select('translate').eq('id', session_id).execute().data
        if sess: do_translate = sess[0].get('translate', TRANSLATE)

    print(f'\n[{done_count+fail_count+1}/{len(jobs)}] {reel_url}')

    db.table('jobs').update({'state': 'in_progress', 'attempts': attempts}).eq('id', job_id).execute()
    db.table('transcripts').update({'status': 'downloading'}).eq('reel_id', reel_id).execute()

    try:
        # скачиваем
        wav_path, info = download_audio(reel_url)
        print(f'  Скачано: {wav_path.name}')

        # транскрибируем
        db.table('transcripts').update({'status': 'transcribing'}).eq('reel_id', reel_id).execute()
        t0 = time.time()
        text, lang, dur = transcribe(wav_path)
        print(f'  Транскрипт ({lang}): {len(text)} симв, {time.time()-t0:.1f}с')

        # переводим
        text_ru = None
        if do_translate and text and lang and lang != 'ru':
            db.table('transcripts').update({'status': 'translating'}).eq('reel_id', reel_id).execute()
            text_ru = translate_to_ru(text)
            print(f'  Перевод: {len(text_ru)} симв')

        # сохраняем
        db.table('transcripts').update({
            'text': text, 'text_ru': text_ru, 'language': lang,
            'duration_sec': int(dur), 'status': 'done', 'fail_reason': None,
        }).eq('reel_id', reel_id).execute()

        db.table('jobs').update({'state': 'done', 'error': None}).eq('id', job_id).execute()
        done_count += 1
        print(f'  ✓ done')

    except Exception as exc:
        traceback.print_exc()
        next_at = (datetime.utcnow() + timedelta(seconds=30 * attempts)).isoformat()
        state = 'failed' if attempts >= MAX_ATTEMPTS else 'queued'
        db.table('jobs').update({
            'state': state, 'attempts': attempts,
            'next_attempt_at': next_at, 'error': str(exc),
        }).eq('id', job_id).execute()
        db.table('transcripts').update({
            'status': 'failed', 'fail_reason': str(exc)[:500],
        }).eq('reel_id', reel_id).execute()
        fail_count += 1
        print(f'  ✗ {exc}')

    finally:
        cleanup()
        time.sleep(random.uniform(1.5, 3.5))  # throttle

print(f'\n=== Готово: {done_count} успешно, {fail_count} ошибок ===')